### Initialize Truveta SDK

In [1]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
%%sparkr
print('hi')

In [2]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
#client = Client(output_mode = OutputMode.PySpark)

study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
# population = study.get_population(title = "Delivery")
#ps-oksug3nqrv5ulhwg7bm7nkqa54
population = study.get_population(title = "Delivery") # change snapshot here
# population

# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()
# snapshot

# Show tables in the snapshot.
snapshot.get_tables()

In [3]:
## Running the decode_concepts function - use for match the concept code with the actual name
from typing import overload
import pyspark.pandas as ps
import pandas as pd
from pyspark.sql import DataFrame

@overload
def decode_concepts(df: pd.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> pd.DataFrame: ...
@overload
def decode_concepts(df: ps.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> ps.DataFrame: ...
@overload
def decode_concepts(df: DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> DataFrame: ...
def decode_concepts(df: pd.DataFrame | ps.DataFrame | DataFrame, drop_concepts: bool = True, columns: list[str] | None = None):
    """
    decodes the top level *ConceptId columns within the given data frame and derives new names (without the ConceptId suffix). 
    It assume that every column ending with `ConceptId` which is an integer or float is a concept column
    :param df the data frame to decode
    :param drop_concepts whether to drop *ConceptId columns (default true)
    :param columns optional direct list of columns to decode disabling the auto infering
    :returns the enhanced data frame
    """
    def should_decode(col: str, dtype: str) -> bool:
        if columns:
            return col in columns
        return col.endswith('ConceptId') and str(dtype) in ('int', 'int32', 'float64', 'float32')
    
    column_names = set(df.columns)

    def target_col(col: str) -> str:
        name = col.removesuffix('ConceptId')
        if name == col and drop_concepts:
            # return the same name
            return name
        while name in column_names:
            name = f"{name}Name"
        return name
    
    def safe_name(name: str) -> str:
        while name in column_names:
            name = f"{name}_tmp"
        return name

    final_order: list[str] = []
    
    if isinstance(df, pd.DataFrame):
        lookup = None
        for col, dtype in zip(df.columns, df.dtypes):
            if should_decode(col, str(dtype)):
                name = target_col(col)
                column_names.add(name)
                if lookup is None:
                    # lazy lookup
                    lookup = spark.sql("SELECT ConceptId, ConceptName FROM Concept").toPandas().set_index("ConceptId").ConceptName
                df[name] = df[col].map(lookup)
                if drop_concepts and name != col:
                    df = df.drop(columns=[col])
                else:
                    final_order.append(col) # keep original
                final_order.append(name)
            else:
                final_order.append(col)
        return df[final_order]

    return_pandas = False
    if isinstance(df, ps.DataFrame):
        return_pandas = True
        df = df.to_spark()
    
    concepts_s = spark.sql("SELECT ConceptId, ConceptName FROM Concept").cache()
    for col, dtype in df.dtypes:
        if should_decode(col, str(dtype)):
            name = target_col(col)
            column_names.add(name)
            if col == name and drop_concepts:
                # need to write to the same name, rename old one
                tmp_name = safe_name(col)
                df = df.withColumnRenamed(col, tmp_name).join(concepts_s.withColumnRenamed("ConceptId", tmp_name).withColumnRenamed("ConceptName", name), on=tmp_name, how="left").drop(tmp_name)
            else:
                df = df.join(concepts_s.withColumnRenamed("ConceptId", col).withColumnRenamed("ConceptName", name), on=col, how="left")
                if drop_concepts:
                    df = df.drop(col)
                else:
                    final_order.append(col)
            final_order.append(name)
        else:
            final_order.append(col)
    df = df.select(final_order)
    return df.pandas_api() if return_pandas else df

In [4]:
def match_code(df, codes_df):
    code_name = decode_concepts(df)
    case_names = codes_df.ConceptName.to_pandas().tolist()
    code_name = code_name[code_name['Code'].isin(case_names)]
    return code_name

### Remove the Ectopic and Multiple Gestation

In [5]:
# getting procedureNotCodes
ectopics_code = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name = "ectopic_code")
ectopics_pro = snapshot.load_filtered_table("Procedure", ectopics_code, view_name = 'tbl_ectopics_pro')
print(ectopics_pro['PersonId'].nunique())
ectopics_con = snapshot.load_filtered_table("Condition", ectopics_code, view_name = 'tbl_ectopics_con')
print(ectopics_con['PersonId'].nunique())

In [6]:
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_ectopics_pro p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
ectopics_pro = match_code(df, ectopics_code)

df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, pm.* FROM tbl_ectopics_con p JOIN ConditionCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
ectopics_con = match_code(df, ectopics_code)
ectopics = pd.concat([ectopics_pro, ectopics_con])
ectopics.PersonId.nunique()

In [7]:
multiple_code = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name = "multiple_code")
multiple_df = snapshot.load_filtered_table("Condition", multiple_code, view_name = 'tbl_multiple')
print(multiple_df['PersonId'].nunique())
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, pm.* FROM tbl_multiple p JOIN ConditionCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
multiple_df = match_code(df, multiple_code)
#multiple_df.PersonId.nunique()

### Getting the Surgery Group

In [8]:
surgerycodes_df = snapshot.codeset_from_prose(url = "/definitions/bariatric-surgery", variable_name = "codes")
index_surgery = snapshot.load_filtered_table("Procedure", surgerycodes_df, view_name = 'tbl_index_surgery')
print(index_surgery['PersonId'].nunique())
#index_surgery.head()
#Check number of unique encounters
index_surgery['EncounterId'].nunique()

In [9]:
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_index_surgery p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
procedure_full = match_code(df, surgerycodes_df)
#procedure_full.head()
# case_names = surgerycodes_df.ConceptName.to_pandas().tolist()
# procedure_full = procedure_full[procedure_full['Code'].isin(case_names)]
#procedure_full.head()
procedure_full['StartDateTime'] = procedure_full['StartDateTime'].fillna(procedure_full['RecordedDateTime'])
procedure_full = procedure_full.drop(columns=['RecordedDateTime'])
procedure_full = procedure_full.dropna()
len(procedure_full), len(procedure_full.PersonId.unique())

In [10]:
from datetime import datetime
procedure_full = procedure_full.sort_values(by=['PersonId', 'StartDateTime']).reset_index(drop=True)

procedure_full['StartDateTime'] = procedure_full['StartDateTime'].dt.date
cutoff_date = datetime.strptime('2017-01-01', '%Y-%m-%d').date()
procedure_full = procedure_full[procedure_full['StartDateTime'] >= cutoff_date]
procedure_full = procedure_full.drop_duplicates(subset=['PersonId', 'StartDateTime'], keep='first').reset_index(drop=True)
print(len(procedure_full.PersonId.unique()))
procedure_full.head()

### Getting the Medication Group

In [11]:
# we are going to use MedicationDispense
medcodes_df = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta/d/semaglutide-medication-code-set", variable_name = "codes")
index_med = snapshot.load_filtered_table("MedicationDispense", medcodes_df, view_name = 'tbl_index_med')
print(index_med['PersonId'].nunique())
#index_med.head()
df = ps.sql("SELECT m.PersonId, m.DaysSupply, m.DispenseDateTime, pm.* FROM tbl_index_med m JOIN MedicationCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
medication_full = match_code(df, medcodes_df)
#medication_full.head()
# case_names = medcodes_df.ConceptName.to_pandas().tolist()
# medication_full = medication_full[medication_full['Code'].isin(case_names)]
medication_full.head()

In [12]:
print(len(medication_full[medication_full.isna().any(axis=1)]))
print(len(medication_full), len(medication_full.PersonId.unique()))
medication_full = medication_full.dropna()
len(medication_full), len(medication_full.PersonId.unique())

In [13]:
medication_full = medication_full.sort_values(by=['PersonId', 'DispenseDateTime']).reset_index(drop=True)

medication_full['DispenseDateTime'] = medication_full['DispenseDateTime'].dt.date
cutoff_date = datetime.strptime('2021-01-01', '%Y-%m-%d').date()
medication_full = medication_full[medication_full['DispenseDateTime'] >= cutoff_date]
medication_full_copy = medication_full.copy()
medication_full = medication_full.drop_duplicates(subset=['PersonId', 'DispenseDateTime'], keep='first').reset_index(drop=True)
len(medication_full.PersonId.unique())

### Finding the Overlap individuals and REMOVE

In [14]:
both_med_surg = procedure_full.merge(medication_full, on='PersonId', how='inner')
print(len(both_med_surg.PersonId.unique()))
# Then, check for rows where the times are equal
#overlap = merged[merged['MedicationTime'] == merged['ProcedureTime']]
# need to drop using both medication and procedure
ect_surgery = procedure_full.merge(ectopics, on='PersonId', how='inner')
print(len(ect_surgery.PersonId.unique()))
ect_med = medication_full.merge(ectopics, on='PersonId', how='inner')
print(len(ect_med.PersonId.unique()))
multi_surgery = procedure_full.merge(multiple_df, on='PersonId', how='inner')
print(len(multi_surgery.PersonId.unique()))
multi_med = medication_full.merge(multiple_df, on='PersonId', how='inner')
print(len(multi_med.PersonId.unique()))

### Getting Weight - Method From the Enablement Study

In [15]:
sql = """
SELECT 
* 
FROM Concept
"""

select_columns = ["ConceptId", "ConceptName"]
#* select columns needed to improve performance
concept = snapshot.load_sql_table(sql, view_name = 'tbl_concept')[["ConceptId", "ConceptName"]]
#display_df(concept)

In [16]:
weight_codes_s  = snapshot.codeset("LOINC", 'selfAndDescendants',
"58229-6",
  "18833-4",
  "29463-7",
  "3141-9",
  "3142-7",
  "8341-0",
  "8349-3",
  "8350-1",
  "8351-9")

study.create_view(weight_codes_s, 
   view_name = 'tbl_weight_codes_s')

In [17]:
lab_weights = snapshot.load_filtered_table("LabResult",
   weight_codes_s, view_name = 'tbl_lab_weights')
   # select and rename columns
lab_weights = lab_weights[['PersonId',
                               'EffectiveDateTime', 
                               'RecordedDateTime', 
                               'NormalizedValueConceptId', 
                               'NormalizedValueNumeric', 
                               'NormalizedValueUOMConceptId', 
                               'StatusConceptId', 
                               'EncounterId']]
display_df(lab_weights)

obs_weights = snapshot.load_filtered_table("Observation",
   weight_codes_s, view_name = 'tbl_obs_weights')

# select and rename columns

# select and rename columns
obs_weights = obs_weights[['PersonId',
                               'EffectiveDateTime', 
                               'RecordedDateTime', 
                               'NormalizedValueConceptId', 
                               'NormalizedValueNumeric', 
                               'NormalizedValueUOMConceptId', 
                               'StatusConceptId', 
                               'EncounterId']]

all_weights = ps.concat([obs_weights, lab_weights], 
   ignore_index=True)


In [18]:
all_weights = all_weights.merge(concept, 
    how='left', 
    left_on='NormalizedValueConceptId', 
    right_on='ConceptId') \
    .rename(columns={'ConceptName': 'NormalizedValueConcept'}) \
    .drop(columns=['ConceptId'])

all_weights = all_weights.merge(concept, 
   how='left', 
   left_on='NormalizedValueUOMConceptId', 
   right_on='ConceptId') \
   .rename(columns={'ConceptName': 'NormalizedValueUOMConcept'}) \
   .drop(columns=['ConceptId'])

all_weights = all_weights.merge(concept, 
   how='left', 
   left_on='StatusConceptId', 
   right_on='ConceptId') \
   .rename(columns={'ConceptName': 'StatusConcept'}) \
   .drop(columns=['ConceptId'])

In [19]:
units_table = all_weights.groupby('NormalizedValueUOMConcept')\
   .size().reset_index()

# take these out
# Define a list of units to filter out
excluded_units = [
    'per liter', 'per meter', 'billion per liter',
    'centimeter', 'degree', 'foot (US)', 'heart beats per minute', 
    'inches', 'liter', 'liter per minute', 'lumen', 
    'meter', 'millimeter mercury column', 'millivolt', 
    'minute', 'per hour', 'per meter', 'per liter', 
    'percent', 'second', 'week', 'Each', 'Inches', 
    'inch (international)', 'each'
]

# Filter the DataFrame
all_weights = all_weights[~all_weights['NormalizedValueUOMConcept'].isin(excluded_units)]

### Make unknowns the same variable: 

unknown_mapping = {
    'No Information': 'unknown',
    'Field has not been mapped': 'unknown',
    'Field is not present in source': 'unknown',
    'Invalid': 'unknown'
}

# Use replace to handle the mapping
all_weights['NormalizedValueUOMConcept'] = \
   all_weights['NormalizedValueUOMConcept'].replace(unknown_mapping)

units_weights_clean = all_weights.groupby('NormalizedValueUOMConcept')\
   .size().reset_index()

lbs_ll =  90
lbs_ul = 700

In [20]:
all_weights['NormalizedValueNumeric'] = \
   all_weights['NormalizedValueNumeric'].astype(float)

# Initialize the new column with NaNs
all_weights['UOM_assumed'] = np.nan

# Conditionally assign values to 'UOM_assumed' using .loc
# when pounds unit then pound
all_weights.loc[
    all_weights['NormalizedValueUOMConcept'].\
    isin(['pound (US and British)', 'pound (US)', 'pound (apothecary)']),
    'UOM_assumed'
] = 'pound'

# If UOM is known and not in the predefined list, use its value
all_weights.loc[
    (all_weights['NormalizedValueUOMConcept'] != 'unknown') &
    (~all_weights['NormalizedValueUOMConcept'].\
    isin(['pound (US and British)', 'pound (US)', 'pound (apothecary)'])),
    'UOM_assumed'
] = all_weights['NormalizedValueUOMConcept']

# Normalize to gram if within the specified range
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] > lbs_ll * 453.6) &
    (all_weights['NormalizedValueNumeric'] <= lbs_ul * 453.6) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'gram'

# Normalize to ounce if within the specified range
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] > lbs_ll * 16) &
    (all_weights['NormalizedValueNumeric'] <= lbs_ul * 16) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'ounce (avoirdupois)'

# Assign kilogram if value is less than 125
all_weights.loc[
    (all_weights['NormalizedValueNumeric'] < 125) &
    (all_weights['NormalizedValueUOMConcept'] == 'unknown'),
    'UOM_assumed'
] = 'kilogram'

# Step 5: Create 'pounds' column based on 'UOM_assumed' 
# and 'NormalizedValueNumeric'
all_weights['pounds'] = np.nan

all_weights.loc[
    all_weights['UOM_assumed'] == 'pound',
    'pounds'
] = all_weights['NormalizedValueNumeric']

all_weights.loc[
    all_weights['UOM_assumed'] == 'ounce (avoirdupois)',
    'pounds'
] = all_weights['NormalizedValueNumeric'] / 16

all_weights.loc[
    all_weights['UOM_assumed'] == 'kilogram',
    'pounds'
] = all_weights['NormalizedValueNumeric'] * 2.205

all_weights.loc[
    all_weights['UOM_assumed'] == 'gram',
    'pounds'
] = all_weights['NormalizedValueNumeric'] / 453.6

#  Mutate 'pounds' column - Set values to NaN if outside the plausible range
all_weights.loc[
    (all_weights['pounds'] < lbs_ll) | (all_weights['pounds'] > lbs_ul),
    'pounds'
] = np.nan

# Create 'kg' column by converting 'pounds' to kilograms
all_weights['kg'] = all_weights['pounds'] / 2.205

all_weights.head()

### Getting Delivery Record and determine Preterm Birth

In [21]:
# delivery condition
delivery_concode = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name= "conditionCodes")
#delivery_concode.head()
delivery_con = snapshot.load_filtered_table("Condition", delivery_concode, view_name = 'tbl_index_delivery_con')
print(delivery_con['PersonId'].nunique())
#delivery_con.head()

df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_delivery_con m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
delivery_con = match_code(df, delivery_concode)
print(len(delivery_con), len(delivery_con.PersonId.unique()))
#delivery_con.head()

# delivery procedure
delivery_procode = snapshot.codeset_from_prose(url = "/definitions/delivery", variable_name= "procedureCodes")
delivery_pro = snapshot.load_filtered_table("Procedure", delivery_procode, view_name = 'tbl_index_delivery_pro')
print(delivery_pro['PersonId'].nunique())
#delivery_pro.head()

df = ps.sql("SELECT m.PersonId, m.StartDateTime, pm.* FROM tbl_index_delivery_pro m JOIN ProcedureCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
delivery_pro = match_code(df, delivery_procode)
delivery_pro.head()

In [ ]:
delivery_concode.head()
delivery_con.head() #ignore OnsetDateTime, just use the RecordedDateTime, use the first instance
delivery_procode.CodeSystem.value_counts()

In [ ]:
delivery_con.Code.value_counts()
delivery_pro.Code.value_counts()

##### Using preterm birth code to define "Preterm Birth" - Don't Run

In [28]:
# getting preterm birth
preterm_concode = snapshot.codeset_from_prose(url = "/definitions/preterm-birth", variable_name= "codes")
preterm_con = snapshot.load_filtered_table("Condition", preterm_concode, view_name = 'tbl_index_preterm')
print(preterm_con['PersonId'].nunique())
#delivery_con.head()

df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_preterm m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
preterm_con = match_code(df, preterm_concode)
print(len(preterm_con), len(preterm_con.PersonId.unique()))
preterm_con.head()

In [29]:
'''
Define Preterm Birth!!
Need to update this part later
'''
#preterm_con.head()
#len(preterm_con[preterm_con.isna().any(axis=1)])
#preterm_con['RecordedDateTime'] = preterm_con['RecordedDateTime'].fillna(preterm_con['OnsetDateTime'])
preterm_con["match_delivery"] = preterm_con['PersonId'].isin(combined_df['PersonId'])
#preterm_con = preterm_con.drop(columns=['OnsetDateTime'])
cutoff_date = datetime.strptime('2022-01-01', '%Y-%m-%d').date()
preterm_con['RecordedDateTime'] = preterm_con['RecordedDateTime'].dt.date
preterm_con = preterm_con[preterm_con['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
print(len(preterm_con.dropna()))#11759 row out of 13934 missing rows
preterm_con = preterm_con.dropna()
#preterm_con.head()
#len(preterm_con), len(preterm_con.PersonId.unique())
preterm_con = preterm_con.sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)
preterm_con = preterm_con.drop_duplicates(subset=['PersonId', 'RecordedDateTime'], keep='first').reset_index(drop=True)
len(preterm_con), len(preterm_con.PersonId.unique())

In [30]:
'''
Identify preterm birth label
Need to update this part later
'''

def is_preterm_related(label):
    label = label.lower()
    if 'preterm' in label or 'premature' in label: #double check here
        return True
    return False

df = delivery_con.copy()
df['is_preterm'] = df['Code'].apply(is_preterm_related)
df = df[df.is_preterm == True].copy()
#df = df.drop(columns=["OnsetDateTime"])
print(len(df), len(df.PersonId.unique()))

df['RecordedDateTime'] = df['RecordedDateTime'].dt.date
df = df[df['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
print(len(df), len(df.PersonId.unique()))
# df['preterm'] = df.PersonId.isin(preterm_con['PersonId'])
# df['preterm'].value_counts()

# merge df with preterm_con
preterm_df = pd.concat([preterm_con, df]).copy()
preterm_df = preterm_df.sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)

In [ ]:
'''
Getting the preterm birth record
Need to update this later
'''

# preterm_df['RecordedDateTime'] = pd.to_datetime(preterm_df['RecordedDateTime']).dt.date
# delivery_df['RecordedDateTime'] = pd.to_datetime(delivery_df['RecordedDateTime']).dt.date
from datetime import timedelta
def match_preterm(row):
    person_id = row['PersonId']
    delivery_date = row['RecordedDateTime']
    
    # find record before delivery
    records = preterm_df[
        (preterm_df['PersonId'] == person_id) &
        (preterm_df['RecordedDateTime'] <= delivery_date + timedelta(days=30)) &
        (preterm_df['RecordedDateTime'] >= delivery_date - timedelta(days=30)) # preterm record within 30 days of the delivery
    ] # should preterm record == delivery_date?
    
    if records.empty:
        return pd.Series([None, None, False])
    
    # keep the record closest to delivery
    closest = records.loc[(delivery_date - records['RecordedDateTime']).idxmin()]
    
    return pd.Series([closest['Code'], closest['RecordedDateTime'], True])

delivery_preterm = delivery_df.copy()
delivery_preterm[['preterm_code', 'preterm_date', 'is_preterm']] = delivery_df.apply(match_preterm, axis=1)

In [33]:
delivery_preterm.is_preterm.value_counts()

#### Merge Delivery condition and procedure together

In [22]:
#ignore OnsetDateTime, just use the RecordedDateTime
#delivery_con['RecordedDateTime'] = delivery_con['RecordedDateTime'].fillna(delivery_con['OnsetDateTime'])

#len(delivery_con[delivery_con.isna().any(axis=1)])
delivery_pro = delivery_pro.rename(columns={'StartDateTime': 'RecordedDateTime'})
combined_df = pd.concat([delivery_con, delivery_pro])
#combined_df.head()
combined_df['RecordedDateTime'] = pd.to_datetime(combined_df['RecordedDateTime'])
# Sort by 'person_id' (ascending) and 'date' (ascending)
combined_df = combined_df.sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)
print(len(combined_df), len(combined_df.PersonId.unique()))
combined_df['RecordedDateTime'] = combined_df['RecordedDateTime'].dt.date
combined_df.head()

In [23]:
# keep only one row their date time once and drop both med&surgery
combined_df_cleaned = combined_df.drop_duplicates(subset=['PersonId', 'RecordedDateTime'], keep='first').reset_index(drop = True)
# drop the overlap ones
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned['PersonId'].isin(both_med_surg['PersonId'])]

# drop the ectopic and multiple pregency
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned['PersonId'].isin(ectopics['PersonId'])]
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned['PersonId'].isin(multiple_df['PersonId'])]
print(len(combined_df_cleaned), len(combined_df_cleaned.PersonId.unique()))

#combined_df_cleaned = combined_df_cleaned.drop(columns=['OnsetDateTime'])
combined_df_cleaned = combined_df_cleaned.dropna()
cutoff_date = datetime.strptime('2022-01-01', '%Y-%m-%d').date()
combined_df_cleaned = combined_df_cleaned[combined_df_cleaned['RecordedDateTime'] >= cutoff_date].reset_index(drop=True)
#combined_df_cleaned = combined_df_cleaned.drop(columns=['index'])
len(combined_df_cleaned), len(combined_df_cleaned.PersonId.unique())

In [24]:
procedure_full = procedure_full.rename(columns={'StartDateTime': 'surgery_date'})
medication_full = medication_full.rename(columns={'DispenseDateTime': 'med_date'})
#merge into a large df contain both med/surgery/delivery information
merged = combined_df_cleaned.merge(procedure_full[['PersonId', 'surgery_date']], on='PersonId', how='left')
merged = merged.merge(medication_full[['PersonId', 'med_date']], on='PersonId', how='left')

merged['is_after_surgery'] = merged['RecordedDateTime'] > merged['surgery_date']
merged['is_after_med'] = merged['RecordedDateTime'] > merged['med_date']

def get_source_type(row):
    if row['is_after_surgery']:
        return 'surgery'
    if row['is_after_med']:
        return 'med'
    return None

merged['source_type'] = merged.apply(get_source_type, axis=1)
#len(merged),len(merged.PersonId.unique())
filtered = merged[merged['source_type'].notna()]

filtered = filtered.sort_values(by=['PersonId', 'RecordedDateTime'])
# keep first delivery after the med/surgery
first_events = filtered.groupby('PersonId').first().reset_index()

delivery_df = first_events[['PersonId', 'RecordedDateTime', 'Code', 'source_type']].copy()
print(len(first_events), len(first_events.PersonId.unique()))
len(merged[merged.source_type == 'surgery'].PersonId.unique()), len(merged[merged.source_type == 'med'].PersonId.unique())

In [25]:
delivery_df.head()

In [37]:
mask = delivery_df['Code'].str.contains('preterm|premature', case=False, na=False) & (~delivery_df['is_preterm'].fillna(False))
missed_preterms_count = mask.sum()
print(f"Missed preterm cases: {missed_preterms_count}")

In [26]:
first_events['event_date'] = first_events['surgery_date'].fillna(first_events['med_date'])
# first_events.head()
delivery_df = first_events[['PersonId', 'RecordedDateTime', 'Code', 'source_type', 'event_date']].copy()
# event_date - med/surgery
delivery_df.head()

In [27]:
def is_preterm_related(label):
    label = label.lower()
    if 'preterm' in label or 'premature' in label: #double check here
        return True
    return False

df = delivery_df.copy()
df['is_preterm'] = df['Code'].apply(is_preterm_related)
df = df[df.is_preterm == True].copy()
#df = df.drop(columns=["OnsetDateTime"])
print(len(df), len(df.PersonId.unique()))

In [28]:
print(len(delivery_df), len(delivery_df.PersonId.unique()))
delivery_df = delivery_df[~delivery_df['PersonId'].isin(df['PersonId'])]
print(len(delivery_df), len(delivery_df.PersonId.unique()))

### Calculate the  using Z-code

In [ ]:
'''
estimate the conception date
Also need to update this later
Don't Run - using the Z code to calculate the estimated_conception_date
'''
def estimate_conception_date(row):
    delivery_date = row['RecordedDateTime']
    # preterm birth 35 weeks, normal 39 weeks
    weeks = 35 if row['is_preterm'] else 39
    return delivery_date - pd.Timedelta(weeks=weeks)

delivery_df['estimated_conception_date'] = delivery_df.apply(estimate_conception_date, axis=1)
delivery_df.head()

In [29]:
# Z code
zcodecode = snapshot.codeset_from_prose(url = "/definitions/pregnancy-zcode", variable_name= "zcodes")
zcode = snapshot.load_filtered_table("Condition", zcodecode, view_name = 'tbl_index_zcodes')
print(zcode['PersonId'].nunique())
df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_zcodes m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
zcode = match_code(df, zcodecode)
print(len(zcode), len(zcode.PersonId.unique()))

In [30]:
print(len(delivery_df), delivery_df.PersonId.nunique())
mask_remove = zcode["Code"].str.lower().isin([
    "weeks of gestation of pregnancy not specified",
    "less than 8 weeks gestation of pregnancy"
])

zcode = zcode.loc[~mask_remove].copy()
zcode["gestational_week_zcode"] = (
    zcode["Code"]
    .str.extract(r"(\d+)", expand=False)
    .astype("float")
)

zcode = (
    zcode.sort_values(["PersonId", "RecordedDateTime", "gestational_week_zcode"])
        .groupby(["PersonId", "RecordedDateTime"], as_index=False)
        .tail(1)
)
zcode.gestational_week_zcode.describe()

In [31]:
zcode = zcode.rename(columns={"RecordedDateTime": "zcodetime"})
zcode = zcode.rename(columns={"Code": "zcode"})
deliv_zcode = delivery_df.merge(zcode, on='PersonId', how='inner')
print(len(deliv_zcode), deliv_zcode.PersonId.nunique())
deliv_zcode = deliv_zcode.sort_values(["PersonId", "zcodetime"])
print(deliv_zcode.isna().sum())

deliv_zcode["RecordedDateTime"] = pd.to_datetime(deliv_zcode["RecordedDateTime"], errors="coerce")
deliv_zcode["zcodetime"] = pd.to_datetime(deliv_zcode["zcodetime"], errors="coerce")

deliv_zcode = deliv_zcode[
    deliv_zcode["zcodetime"] <= deliv_zcode["RecordedDateTime"]
].copy()
deliv_zcode["diff_days"] = (
    deliv_zcode["RecordedDateTime"] - deliv_zcode["zcodetime"]
) / pd.Timedelta(days=1)
deliv_zcode = deliv_zcode[deliv_zcode["diff_days"] <= 300].copy()

deliv_zcode["zcodetime_date"] = deliv_zcode["zcodetime"].dt.date

deliv_zcode = (
    deliv_zcode
    .sort_values(["PersonId", "RecordedDateTime", "zcodetime_date", "gestational_week_zcode"])
    .groupby(["PersonId", "RecordedDateTime", "zcodetime_date"], as_index=False)
    .tail(1)
)

idx = deliv_zcode.groupby(["PersonId", "RecordedDateTime"])["diff_days"].idxmin()
closest_df = deliv_zcode.loc[idx].copy()
print(len(closest_df), closest_df.PersonId.nunique())

zcode_counts = (
    deliv_zcode
    .groupby(["PersonId", "RecordedDateTime"])
    .size()
    .reset_index(name="zcode_count")
)
print(zcode_counts.shape, zcode_counts.PersonId.nunique())
zcode_counts.head()

In [32]:
delivery_df = closest_df.copy()

delivery_df[""] = (
    delivery_df["zcodetime"] -
    pd.to_timedelta(delivery_df["gestational_week_zcode"] * 7, unit="D")
)

delivery_df["gestational_week"] = (
    (delivery_df["RecordedDateTime"] - delivery_df[""])
    / pd.Timedelta(days=7)
)

delivery_df["gestational_age_days_at_delivery"] = (
    delivery_df["RecordedDateTime"] - delivery_df[""]
).dt.days

delivery_df = delivery_df[delivery_df["gestational_week"] >= 24].copy()
delivery_df = delivery_df[delivery_df["gestational_week"] <= 42].copy()
print(delivery_df.gestational_week.describe())

delivery_df["preterm"] = (delivery_df["gestational_week"] < 37).astype(int)
print(delivery_df["preterm"].value_counts())

In [33]:
print(delivery_df.shape)
delivery_df = delivery_df.merge(zcode_counts[['PersonId', 'zcode_count']], on='PersonId', how='inner')

In [34]:
delivery_df.shape

In [35]:
import matplotlib.pyplot as plt
mode_week = delivery_df["gestational_week"].round().mode()[0]
plt.figure()
delivery_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")

plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Distribution of gestational age at delivery")

plt.legend()
plt.show()

In [36]:
# print(result_df.gestational_week.describe(), result_df.gestational_age_days_at_delivery.describe())
# mode_week = result_df["gestational_week"].round().mode()[0]
# median_week = result_df["gestational_week"].round(2).median()
# plt.figure()
# result_df["gestational_week"].hist(bins=40)
# plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
# plt.axvline(median_week, linestyle="--", label=f"Median week: {median_week}")
# plt.xlabel("Gestational age (weeks)")
# plt.ylabel("Count")
# plt.title("Distribution of gestational age at delivery")

# plt.legend()
# plt.show()

# mode_days = result_df["gestational_age_days_at_delivery"].round().mode()[0]
# median_days = result_df["gestational_age_days_at_delivery"].round(2).median()
# plt.figure()
# result_df["gestational_age_days_at_delivery"].hist(bins=40)
# plt.axvline(mode_days, linestyle="--", label=f"Most common week: {mode_days}")
# plt.axvline(median_days, linestyle="--", label=f"Median week: {median_days}")
# plt.xlabel("Gestational age (days)")
# plt.ylabel("Count")
# plt.title("Distribution of gestational age at delivery")

# plt.legend()
# plt.show()

In [37]:
full_med = delivery_df[delivery_df["source_type"] == "med"].copy()

### Checking Pregnancy Date and adding on Weight

In [38]:
ps.set_option("compute.ops_on_diff_frames", True)

In [39]:
'''
getting the weight record:
1. pre-pregancy - within 12 weeks before or after the date
2. pre-delivery - within 4 weeks before delivery date
'''

from pyspark.sql import functions as F
from pyspark.sql import Window

all_weights_df = all_weights[['PersonId', 'RecordedDateTime', 'pounds', 'kg']].copy()
all_weights_df['RecordedDateTime'] = all_weights_df['RecordedDateTime'].dt.date
all_weights_df = all_weights_df.dropna()
weights_df = all_weights_df.to_spark()
weights_df = weights_df.withColumn("RecordedDateTime", F.to_timestamp("RecordedDateTime"))
delivery_sdf = spark.createDataFrame(delivery_df).withColumn("RecordedDateTime", F.to_timestamp("RecordedDateTime"))

# find match PersonId in both weight and delivery
person_ids_with_weight = weights_df.select("PersonId").distinct()
person_ids_with_delivery = delivery_sdf.select("PersonId").distinct()

# get those person
valid_person_ids = person_ids_with_weight.join(person_ids_with_delivery, on="PersonId", how="inner")

# filteriong out both side of the data
delivery_sdf = delivery_sdf.join(valid_person_ids, on="PersonId", how="inner")
weights_df = weights_df.join(valid_person_ids, on="PersonId", how="inner")

all_weights_df.head()

In [40]:
delivery_df.head()

In [ ]:
#weights_df.count(), weights_df.select("PersonId").distinct().count()

In [41]:
all_weights_df_pd = all_weights_df.to_pandas()
all_weights_df_pd.head()

In [42]:
import pandas as pd
from datetime import timedelta

def add_weight_info(delivery_df, all_weights_df):
    delivery_df = delivery_df.copy()
    all_weights_df = all_weights_df.copy()

    # Ensure datetime
    delivery_df["RecordedDateTime"] = pd.to_datetime(delivery_df["RecordedDateTime"], errors="coerce")
    delivery_df["event_date"] = pd.to_datetime(delivery_df["event_date"], errors="coerce")
    delivery_df[""] = pd.to_datetime(delivery_df[""], errors="coerce")

    all_weights_df["RecordedDateTime"] = pd.to_datetime(all_weights_df["RecordedDateTime"], errors="coerce")

    # Drop rows with missing keys needed for windows
    delivery_df = delivery_df.dropna(subset=["PersonId", "RecordedDateTime", "event_date", ""])
    all_weights_df = all_weights_df.dropna(subset=["PersonId", "RecordedDateTime", "kg"])

    # storage result
    weight_around_lmp = []
    has_weight_around_lmp = []
    weight_before_del = []
    has_weight_before_del = []
    latest_weight_before_event = []
    months_between_event_and_lmp = []

    delivery_df = delivery_df.rename(columns={"": "lmp_date"})

    for row in delivery_df.itertuples(index=False):
        person_id = row.PersonId
        lmp = pd.Timestamp(row.lmp_date)
        delivery = pd.Timestamp(row.RecordedDateTime)
        event_date = pd.Timestamp(row.event_date)

        person_weights = all_weights_df[all_weights_df["PersonId"] == person_id]

        # weight around LMP (+/- 12 weeks, excluding exact LMP if you want)
        start_lmp = lmp - timedelta(weeks=12)
        end_lmp = lmp + timedelta(weeks=12)

        con_window = person_weights[
            ((person_weights["RecordedDateTime"] >= start_lmp) & (person_weights["RecordedDateTime"] < lmp)) |
            ((person_weights["RecordedDateTime"] > lmp) & (person_weights["RecordedDateTime"] <= end_lmp))
        ]

        if not con_window.empty:
            nearest = con_window.loc[(con_window["RecordedDateTime"] - lmp).abs().idxmin()]
            weight_around_lmp.append(nearest["kg"])
            has_weight_around_lmp.append(True)
        else:
            weight_around_lmp.append(pd.NA)
            has_weight_around_lmp.append(False)

        # predelivery weight within 4 weeks before delivery date
        start_del = delivery - timedelta(weeks=4)
        del_window = person_weights[
            (person_weights["RecordedDateTime"] >= start_del) &
            (person_weights["RecordedDateTime"] <= delivery)
        ]

        if not del_window.empty:
            nearest_del = del_window.loc[(del_window["RecordedDateTime"] - delivery).abs().idxmin()]
            weight_before_del.append(nearest_del["kg"])
            has_weight_before_del.append(True)
        else:
            weight_before_del.append(pd.NA)
            has_weight_before_del.append(False)

        # pretreatment weight within 6 months before event_date
        six_months_before_event = event_date - timedelta(days=183)
        before_event = person_weights[
            (person_weights["RecordedDateTime"] >= six_months_before_event) &
            (person_weights["RecordedDateTime"] <= event_date)
        ]

        if not before_event.empty:
            nearest_before_event = before_event.loc[(before_event["RecordedDateTime"] - event_date).abs().idxmin()]
            latest_weight_before_event.append(nearest_before_event["kg"])
        else:
            latest_weight_before_event.append(pd.NA)

        # months between event and LMP (positive means LMP after event)
        delta_days = (lmp - event_date).days
        months_between_event_and_lmp.append(round(delta_days / 30.44, 1))

    # add to df
    delivery_df["has_prepreg_weight"] = has_weight_around_lmp
    delivery_df["prepreg_weight"] = weight_around_lmp
    delivery_df["has_predelivery_weight"] = has_weight_before_del
    delivery_df["predelivery_weight"] = weight_before_del
    delivery_df["pretreatment_weight"] = latest_weight_before_event
    delivery_df["months_between_event_and_conception"] = months_between_event_and_lmp

    # keep only those with pretreatment_weight
    delivery_df = delivery_df[delivery_df["pretreatment_weight"].notna()].copy()

    return delivery_df

In [115]:
'''DON'T RUN THIS, main purpose for checking the pretreatment numbers'''

# def add_weight_info1(delivery_df, all_weights_df):
#     # make sure it is the datetime type
#     # delivery_df['RecordedDateTime'] = pd.to_datetime(delivery_df['RecordedDateTime'])
#     # delivery_df['estimated_conception_date'] = pd.to_datetime(delivery_df['estimated_conception_date'])
#     # all_weights_df['RecordedDateTime'] = pd.to_datetime(all_weights_df['RecordedDateTime'])

#     # storage result
#     weight_around_con = [] # weight for prepreg - unit kg
#     has_weight_around_con = [] 
#     weight_before_del = [] # weight before delivery
#     has_weight_before_del = []

#     latest_weight_before_event = []  # weight before treatment
#     months_between_event_and_conception = []  

#     for row in delivery_df.itertuples():
#         person_id = row.PersonId
#         conception = row.estimated_conception_date
#         delivery = row.RecordedDateTime
#         event_date = row.event_date

#         person_weights = all_weights_df_pd[all_weights_df_pd['PersonId'] == person_id]

#         # check between estimated_conception_date +- 12 week "OR"
#         start_con = conception - timedelta(weeks=12)
#         end_con = conception + timedelta(weeks=12)
#         con_window = person_weights[
#         ((person_weights['RecordedDateTime'] >= start_con) & (person_weights['RecordedDateTime'] < conception)) |
#         ((person_weights['RecordedDateTime'] > conception) & (person_weights['RecordedDateTime'] <= end_con))]
#         if not con_window.empty: # if have value
#         # keep the one close to conception(the estimated_conception_date)
#             nearest_con = con_window.loc[(con_window['RecordedDateTime'] - conception).abs().idxmin()]
#             weight_around_con.append(nearest_con['kg'])
#             has_weight_around_con.append(True)
#         else:
#             weight_around_con.append(None)
#             has_weight_around_con.append(False)

#         # check for the predelivery weight, 4 weeks util delivery date
#         start_del = delivery - timedelta(weeks=4)
#         del_window = person_weights[
#             (person_weights['RecordedDateTime'] >= start_del) &
#             (person_weights['RecordedDateTime'] <= delivery)
#         ]
#         if not del_window.empty:
#             nearest_del = del_window.loc[(del_window['RecordedDateTime'] - delivery).abs().idxmin()]
#             weight_before_del.append(nearest_del['kg'])
#             has_weight_before_del.append(True)
#         else:
#             weight_before_del.append(None)
#             has_weight_before_del.append(False)

#         # find weight within 6 months before the event_date (pretreatment)
#         six_months_before_event = event_date - timedelta(days=183)
#         before_event = person_weights[
#             (person_weights['RecordedDateTime'] >= six_months_before_event) &
#             (person_weights['RecordedDateTime'] <= event_date)
#         ]
#         if not before_event.empty:
#             nearest_before_event = before_event.loc[(before_event['RecordedDateTime'] - event_date).abs().idxmin()]
#             latest_weight_before_event.append(nearest_before_event['kg'])
#         else:
#             latest_weight_before_event.append(None)

#         # get the interval event_date and estimated_conception_date (month)
#         delta_days = (conception - event_date).days
#         months_between_event_and_conception.append(round(delta_days / 30.44, 1)) 

#     # add those to df
#     delivery_df['has_prepreg_weight'] = has_weight_around_con
#     delivery_df['prepreg_weight'] = weight_around_con
#     delivery_df['has_predelivery_weight'] = has_weight_before_del
#     delivery_df['predelivery_weight'] = weight_before_del
#     delivery_df['pretreatment_weight'] = latest_weight_before_event
#     delivery_df['months_between_event_and_conception'] = months_between_event_and_conception

#     delivery_df['pretreatment_weight_missing'] = delivery_df['pretreatment_weight'].isna()


#     return delivery_df

# random = add_weight_info1(delivery_df, all_weights_df_pd)
# random.PersonId.nunique()

# count_sur = random[delivery_df['pretreatment_weight'].isna() & (random['source_type'] == 'surgery')].shape[0]
# count_med = random[delivery_df['pretreatment_weight'].isna() & (random['source_type'] == 'med')].shape[0]
# print(count_sur, count_med)
# count_sur = random[(random['pretreatment_weight'].isna()) & (random['source_type'] == 'surgery')].shape[0]
# count_med = random[(random['pretreatment_weight'].isna()) & (random['source_type'] == 'med')].shape[0]
# print(count_sur, count_med)
# print(random[(random['has_prepreg_weight'] == False) & (random['source_type'] == 'surgery')].shape[0], 
# random[(random['has_prepreg_weight'] == False) & (random['source_type'] == 'med')].shape[0])
# print(random[(random['has_predelivery_weight'] == False) & (random['source_type'] == 'surgery')].shape[0], 
# random[(random['has_predelivery_weight'] == False) & (random['source_type'] == 'med')].shape[0])
    #delivery_df_pd['has_predelivery_weight']

In [43]:
#delivery_df[''] = pd.to_datetime(delivery_df[''])

delivery_df_pd = add_weight_info(delivery_df, all_weights_df_pd)
# delivery_df_pd = delivery_df[['PersonId', 'RecordedDateTime', 'Code', 'source_type', 'preterm_code',
#        'preterm_date', 'is_preterm', 'estimated_LMP',
#        'has_prepreg_weight', 'prepreg_weight', 'has_predelivery_weight',
#        'predelivery_weight']].copy()
delivery_df_pd.head()
print(delivery_df_pd['has_prepreg_weight'].value_counts(), delivery_df_pd['has_predelivery_weight'].value_counts())

In [44]:
delivery_df_pd['has_both_weights'] = (
    delivery_df_pd['has_prepreg_weight'] &
    delivery_df_pd['has_predelivery_weight']
)
delivery_df_pd = delivery_df_pd[delivery_df_pd['has_both_weights'] == True].copy()
delivery_df_pd['gestation_weight'] = (
    delivery_df_pd['predelivery_weight'] - delivery_df_pd['prepreg_weight'])
    
delivery_df_pd.head()

In [45]:
delivery_df_pd.shape

In [46]:
delivery_df_pd.gestational_week.describe(), delivery_df_pd.preterm.value_counts()

In [51]:
all_weights_df_pd.shape

#### Adding height or BMI

In [47]:
bmi_df = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta/d/tr-body-mass-index-bmi", variable_name = "codes")
index_bmi_df = snapshot.load_filtered_table("Observation", bmi_df, view_name = 'tbl_index_bmi')
print(index_bmi_df['PersonId'].nunique())
index_bmi_df_lab = snapshot.load_filtered_table("LabResult", bmi_df, view_name = 'tbl_index_bmilab')
#index_bmi_df_lab

In [48]:
index_bmi_df["RecordedDateTime"] = pd.to_datetime(index_bmi_df["RecordedDateTime"]).astype("datetime64[ns]")
index_bmi_df = ps.from_pandas(index_bmi_df)
index_bmi_df = ps.concat([index_bmi_df, index_bmi_df_lab])

In [ ]:
index_bmi_df = index_bmi_df[['NormalizedValueNumeric', 'PersonId', 'RecordedDateTime']].sort_values(by=['PersonId', 'RecordedDateTime']).reset_index(drop=True)

# Drop rows with missing BMI or RecordedDateTime
index_bmi_df = index_bmi_df.dropna(subset=['NormalizedValueNumeric', 'RecordedDateTime'])
index_bmi_df = index_bmi_df.drop_duplicates(subset=['PersonId', 'RecordedDateTime'], keep='first')
index_bmi_df = index_bmi_df.to_pandas()
# Convert date column to datetime type
index_bmi_df['RecordedDateTime'] = ps.to_datetime(index_bmi_df['RecordedDateTime'])
len(index_bmi_df.PersonId.unique())

In [ ]:
delivery_df_pd = delivery_df_pd.rename(columns={'RecordedDateTime': 'delivery_date'})
# index_bmi_df = index_bmi_df.to_pandas()
# change bmi to numeric type
index_bmi_df['NormalizedValueNumeric'] = (
    index_bmi_df['NormalizedValueNumeric']
    .astype(float)
)

# remove nan or close to 0
index_bmi_df = index_bmi_df.dropna(subset=['NormalizedValueNumeric'])
index_bmi_df = index_bmi_df[index_bmi_df['NormalizedValueNumeric'] > 10.0]

In [ ]:
merged_df = pd.merge(index_bmi_df, delivery_df_pd, on='PersonId')
# Pre-pregnancy BMI: BMI before conception

# change date to datetime type
merged_df['RecordedDateTime'] = pd.to_datetime(merged_df['RecordedDateTime'])
merged_df[''] = pd.to_datetime(merged_df[''])
merged_df['delivery_date'] = pd.to_datetime(merged_df['delivery_date'])

pre_bmi_df = merged_df.copy()
# find the closest time difference 
pre_bmi_df['TimeDiff'] = (pre_bmi_df[''] - pre_bmi_df['RecordedDateTime']).abs()
pre_bmi_df = pre_bmi_df.sort_values(['PersonId', 'TimeDiff']).drop_duplicates('PersonId', keep='first')[
    ['PersonId', 'NormalizedValueNumeric']
].rename(columns={'NormalizedValueNumeric': 'PrePregnancyBMI'})


# Post-delivery BMI: first after delivery
post_bmi_df = merged_df.copy()
# find the closest bmi
post_bmi_df['TimeDiff'] = (post_bmi_df['RecordedDateTime'] - post_bmi_df['delivery_date']).abs()

post_bmi_df = post_bmi_df.sort_values(['PersonId', 'TimeDiff']).drop_duplicates('PersonId', keep='first')[
    ['PersonId', 'NormalizedValueNumeric']
].rename(columns={'NormalizedValueNumeric': 'nearDeliveryBMI'})

In [ ]:
# pre Treatment BMI: first after delivery
pretreat_df = merged_df.copy()
# find the closest bmi
pretreat_df['TimeDiff'] = (pretreat_df['RecordedDateTime'] - pretreat_df['delivery_date']).abs()

pretreat_df = pretreat_df.sort_values(['PersonId', 'TimeDiff']).drop_duplicates('PersonId', keep='first')[
    ['PersonId', 'NormalizedValueNumeric']
].rename(columns={'NormalizedValueNumeric': 'preTreatmentBMI'})

In [ ]:
#delivery_df_pd = delivery_df_pd.drop(columns=['PrePregnancyBMI', 'nearDeliveryBMI'], errors='ignore')

delivery_df_pd = delivery_df_pd.merge(pre_bmi_df, on='PersonId', how='left')

# Merge post-delivery BMI
delivery_df_pd = delivery_df_pd.merge(post_bmi_df, on='PersonId', how='left')
delivery_df_pd = delivery_df_pd.merge(pretreat_df, on='PersonId', how='left')

In [ ]:
delivery_df_pd.nearDeliveryBMI.describe()

In [ ]:
delivery_df_pd.head()

In [ ]:
#all_weights_df_pd[all_weights_df_pd.PersonId == '01ed93e9-1217-1d05-4857-f6ade2e07a1a']
print(delivery_df_pd['pretreatment_weight'].isna().sum())
delivery_df_pd.months_between_event_and_conception.median()

In [ ]:
len(delivery_df_pd), delivery_df_pd.gestation_weight.mean(), delivery_df_pd.gestation_weight.median()

In [ ]:
len(delivery_df_pd[delivery_df_pd.source_type=='med']), len(delivery_df_pd[delivery_df_pd.source_type=='surgery'])

In [ ]:
delivery_df_pd[delivery_df_pd.source_type=='med'].gestation_weight.median(), delivery_df_pd[delivery_df_pd.source_type=='surgery'].gestation_weight.median()

In [46]:
all_weights_df_pd[all_weights_df_pd.PersonId == "041f59e2-3836-07bd-f7e3-86833f6e7601"]#.gestation_weight.median()

In [ ]:
delivery_df_pd.shape

In [ ]:
delivery_df_pd.source_type.value_counts()

In [59]:
zcode_count_df = delivery_df_pd[['PersonId', 'zcode_count']].copy()

In [60]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/test_zcodecount.csv"
zcode_count_df.to_csv(file_to_write, index = False)

### Outcome variables - Table 3

In [61]:
'''
if wanted this can be apply in the beginning, but for test purpose, just keep it simple
'''
def load_condition_data(
    snapshot,
    codeset_url=None,
    code_set=None,
    codes = 'codes',
    table_name="Condition",
    view_name="tbl_index_condition",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId",
    verbose=True
):
    """
    parameter:
        snapshot: Truveta snapshot 
        codeset_url: if use prose URL load codeset, fill out this
        code_set: if use manual input code snapshot.codeset(...) use this
        table_name: 'Condition', 'Procedure'
        view_name: sql table
        concept_map_table: mapping 'ConditionCodeConceptMap'
        concept_map_key: mapping key 'CodeConceptMapId'
        verbose: print out?
    
    return:
        matched_df: after mathcing pandas DataFrame
        unique_person_count: counting number of PersonId
    """
    # support two ways from prose URL load the code_set
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    # temp table
    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)

    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):", index_table['PersonId'].nunique())

    # SQL match code
    sql_query = f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.* 
        FROM {view_name} m 
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
    """
    df = ps.sql(sql_query).to_pandas()

    # getting the code
    matched_df = match_code(df, code_set)

    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df['PersonId'].nunique())

    return matched_df, matched_df['PersonId'].nunique()

In [62]:
# Gestational diabetes
# defGestationalDiabetes = import "https://library.truveta.com/o/truveta-research/d/gestational-diabetes" change to this one
# gest_diabet_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/gestational-diabetes", variable_name= "codes")
# index_gest_diabet = snapshot.load_filtered_table("Condition", gest_diabet_code, view_name = 'tbl_index_gest_diabet')
# print(index_gest_diabet['PersonId'].nunique())

# df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_gest_diabet m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
# gest_diabet = match_code(df, gest_diabet_code)
# print(len(gest_diabet), len(gest_diabet.PersonId.unique()))

url = "https://library.truveta.com/o/truveta-research/d/gestational-diabetes"
gest_diabet, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_gest_diabet",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [63]:
# defType2Diabetes = include "/definitions/type-2-diabetes"
url = "/definitions/type-2-diabetes"
type2_diabet, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_type2_diabet",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
type2_diabet.head()

In [64]:
# Gestational hypertension 
# defTrGestationalHypertension = import "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension"
#gest_hyper_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension", variable_name= "codes")
url = "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension"
gest_hyper, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_gest_hyper",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [65]:
# defHypertension = include "/definitions/hypertension"
url = "/definitions/hypertension"
hypertension, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    codes = 'conditionCodes', 
    view_name="tbl_index_hyer",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
hypertension.head()

In [66]:
# Preeclampsia 
# defTrPreeclampsia = import "https://library.truveta.com/o/truveta-research/d/tr-preeclampsia"
#preeclampsia_code = snapshot.codeset_from_prose(url = "https://library.truveta.com/o/truveta-research/d/tr-preeclampsia", variable_name= "codes")
url =  "/definitions/preeclampsia"
preeclampsia, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_preeclampsia",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [67]:
# C-section
# defTrCesareanDeliveryProcedures = import "https://library.truveta.com/o/truveta-research/d/tr-cesarean-delivery-procedures"

csection_code = snapshot.codeset_from_prose(url = "/definitions/c-section", variable_name = "codes")
index_c = snapshot.load_filtered_table("Procedure", csection_code, view_name = 'tbl_index_c')
print(index_c['PersonId'].nunique())
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_index_c p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
csection = match_code(df, csection_code)
#procedure_full.head()
# case_names = surgerycodes_df.ConceptName.to_pandas().tolist()
# procedure_full = procedure_full[procedure_full['Code'].isin(case_names)]
#procedure_full.head()
csection['StartDateTime'] = csection['StartDateTime'].fillna(csection['RecordedDateTime'])
csection = csection.drop(columns=['RecordedDateTime'])
csection = csection.rename(columns={'StartDateTime': 'RecordedDateTime'})
csection.head()

In [68]:
defStillbirthCodeSet = "https://library.truveta.com/o/truveta/d/stillbirth-code-set"
stillbirth, count = load_condition_data(
    snapshot, 
    codeset_url=defStillbirthCodeSet,
    table_name="Condition",
    view_name="tbl_index_sb",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [69]:
# # Large-for-gestational-age infants P08.0, P08.1
# large_gest_age = snapshot.codeset('ICD10CM', 'self', 'P08.0', 'P08.1')
# large_gest, count = load_condition_data(
#     snapshot,
#     code_set=large_gest_age,
#     table_name="Condition",
#     view_name="tbl_index_large_gest_age"
# )

# # Small-for-gestational-age infants	P05.1
# small_gest_age = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'P05.1') # self or selfandascendent
# small_gest, count = load_condition_data(
#     snapshot,
#     code_set=small_gest_age,
#     table_name="Condition",
#     view_name="tbl_index_small_gest_age"
# )

defExcessiveFetalWeight = "/definitions/excessive-fetal-weight"
excessive_fetal_weight, count = load_condition_data(
    snapshot, 
    codeset_url=defExcessiveFetalWeight,
    table_name="Condition",
    view_name="tbl_index_efw",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

# Intrauterine growth restriction O36.59
intra_grow_restrict_code = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'O36.59', 'Z36.4', 'O36.5990', 'O36.591', 'O36.592', 'O36.593', 'O36.599') # self or selfandascendent
intra_grow_restrict, count = load_condition_data(
    snapshot,
    code_set=intra_grow_restrict_code,
    table_name="Condition",
    view_name="tbl_index_intra_grow_restrict"
)

In [70]:
#Parity primiparous
primiparous = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'Z34.00', 'O09.611', 'O09.511', 'O09.512', 'O09.513') # self or selfandascendent
primiparous, count = load_condition_data(
    snapshot,
    code_set=primiparous,
    table_name="Condition",
    view_name="tbl_index_primiparous"
)

#Parity multiparous
multiparous = snapshot.codeset('ICD10CM', 'selfAndDescendants', 'O34.21', 'O09.521', 'O09.40', 'O09.621', 'O09.41', 'O09.42', 'O09.43') # self or selfandascendent
multiparous, count = load_condition_data(
    snapshot,
    code_set=multiparous,
    table_name="Condition",
    view_name="tbl_index_multiparous"
)

In [77]:
# check if those condition happens during pregnancy
def mark_condition_in_pregnancy(condition_df, delivery_df, col_name):
    """
    check if condition happen in preg, mark T/F to condition_col_name
    parameter:
        condition_df: condition record - table 3 variables
        delivery_df: PersonId, estimated_LMP, delivery_date, record for delivery/preg
        col_name
        
    return updated delivery_df with new col T/F
    """
    merged = condition_df.merge(
        delivery_df[['PersonId', 'estimated_LMP', 'delivery_date']],
        on='PersonId',
        how='left'
    )

    #check
    merged['in_pregnancy'] = (
        (merged['RecordedDateTime'] >= merged['estimated_LMP']) &
        (merged['RecordedDateTime'] <= merged['delivery_date'])
    )

    # keep satisfied PersonId
    flagged_ids = merged.loc[merged['in_pregnancy'], 'PersonId'].drop_duplicates()
    condition_flag = pd.DataFrame({ 'PersonId': flagged_ids, col_name: True })

    # merge back delivery_df others are False
    delivery_df = delivery_df.merge(condition_flag, on='PersonId', how='left')
    delivery_df[col_name] = delivery_df[col_name].fillna(False)

    return delivery_df

In [72]:
#defHyperlipidemia = import "https://library.truveta.com/o/truveta-research/d/hyperlipidemia"
url = "https://library.truveta.com/o/truveta-research/d/hyperlipidemia"
hyperlipidemia, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_hyerlip",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
hyperlipidemia.head()

In [73]:
defPrenatal = "/definitions/prenatal"
Prenatal, count = load_condition_data(
    snapshot, 
    codeset_url=defPrenatal,
    table_name="Condition",
    codes="Prenatal",
    view_name="tbl_index_Prenatal",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)

In [74]:
# defObstructiveSleepApnea = import "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
url = "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea"
osa, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    codes="osaCodes",
    view_name="tbl_index_osa",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
osa.head()


In [75]:
#defMajorDepression = import "https://library.truveta.com/o/truveta-research/d/major-depression"
url = "https://library.truveta.com/o/truveta-research/d/major-depression"
depression, count = load_condition_data(
    snapshot, 
    codeset_url=url,
    table_name="Condition",
    view_name="tbl_index_mdep",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
depression.head()

In [ ]:
#small_gest_age TEST
df_con = snapshot.load_filtered_table("Condition", small_gest_age, view_name = 'tbl_index_df')
print(df_con['PersonId'].nunique())
#delivery_con.head()

df = ps.sql("SELECT m.PersonId, m.RecordedDateTime, pm.* FROM tbl_index_preterm m JOIN ConditionCodeConceptMap pm on m.CodeConceptMapId = pm.Id").to_pandas()
df_con = match_code(df, small_gest_age)
print(len(df_con), len(df_con.PersonId.unique()))

In [79]:
full_med = full_med.rename(columns={
    'RecordedDateTime': 'delivery_date'
})

In [80]:
full_med = mark_condition_in_pregnancy(gest_diabet, full_med, 'gest_diabet')
full_med = mark_condition_in_pregnancy(gest_hyper, full_med, 'gest_hyper')
full_med = mark_condition_in_pregnancy(preeclampsia, full_med, 'preeclampsia')
full_med = mark_condition_in_pregnancy(csection, full_med, 'csection')
full_med = mark_condition_in_pregnancy(excessive_fetal_weight, full_med, 'excessive_fetal_weight')
full_med = mark_condition_in_pregnancy(intra_grow_restrict, full_med, 'intra_grow_restrict')
full_med = mark_condition_in_pregnancy(Prenatal, full_med, 'obstetric_care')

In [95]:
delivery_df_t3 = mark_condition_in_pregnancy(gest_diabet, delivery_df_pd, 'gest_diabet')
delivery_df_t3 = mark_condition_in_pregnancy(gest_hyper, delivery_df_t3, 'gest_hyper')
delivery_df_t3 = mark_condition_in_pregnancy(preeclampsia, delivery_df_t3, 'preeclampsia')
delivery_df_t3 = mark_condition_in_pregnancy(csection, delivery_df_t3, 'csection')
delivery_df_t3 = mark_condition_in_pregnancy(excessive_fetal_weight, delivery_df_t3, 'excessive_fetal_weight')
# delivery_df_t3 = mark_condition_in_pregnancy(small_gest, delivery_df_t3, 'small_gest_age')
delivery_df_t3 = mark_condition_in_pregnancy(intra_grow_restrict, delivery_df_t3, 'intra_grow_restrict')
delivery_df_t3 = mark_condition_in_pregnancy(Prenatal, delivery_df_t3, 'obstetric_care')

delivery_df_t3.head()

In [81]:
prior_Csection = snapshot.codeset("ICD10CM",
  "selfAndDescendants",
  "O34.212",
  "O34.211",
  "O34.21",
  "O34.219") # self or selfandascendent
prior_Csection, count = load_condition_data(
    snapshot,
    code_set=prior_Csection,
    table_name="Condition",
    view_name="tbl_index_prior_Csection"
)

Prior_Preterm_Birth = snapshot.codeset("ICD10CM",
  "selfAndDescendants","Z87.51",
  "O09.21",
  "O09.211",
  "O09.212",
  "O09.213",
  "O09.219",) # self or selfandascendent
Prior_Preterm_Birth, count = load_condition_data(
    snapshot,
    code_set=Prior_Preterm_Birth,
    table_name="Condition",
    view_name="tbl_index_Prior_Preterm_Birth"
)


# delivery_df_t3["prior_Csection"] = delivery_df_t3["PersonId"].isin(prior_Csection["PersonId"])
# delivery_df_t3["Prior_Preterm_Birth"] = delivery_df_t3["PersonId"].isin(Prior_Preterm_Birth["PersonId"])

full_med["prior_Csection"] = full_med["PersonId"].isin(prior_Csection["PersonId"])
full_med["Prior_Preterm_Birth"] = full_med["PersonId"].isin(Prior_Preterm_Birth["PersonId"])

In [97]:
delivery_df_t3['delivery_type'] = np.where(delivery_df_t3['csection'] == True, 'C-section', 'Vaginal')

# # baby weight
# conditions = [
#     delivery_df_t3['large_gest_age'] == True,
#     delivery_df_t3['small_gest_age'] == True
# ]
# choices = ['Large', 'Small']

# delivery_df_t3['infant_gest_age_class'] = np.select(conditions, choices, default='Average')
delivery_df_t3['infant_gest_age_class'] = np.where(delivery_df_t3['excessive_fetal_weight'] == True, 'Excessive', 'Average')
delivery_df_t3.infant_gest_age_class.value_counts(), delivery_df_t3.delivery_type.value_counts()

In [82]:
full_med['delivery_type'] = np.where(full_med['csection'] == True, 'C-section', 'Vaginal')
full_med['infant_gest_age_class'] = np.where(full_med['excessive_fetal_weight'] == True, 'Excessive', 'Average')
full_med.infant_gest_age_class.value_counts(), full_med.delivery_type.value_counts()

In [98]:
delivery_df_t3['parity'] = 'Unknown'
delivery_df_t3.loc[delivery_df_t3['PersonId'].isin(primiparous['PersonId']), 'parity'] = 'Primiparous'
delivery_df_t3.loc[delivery_df_t3['PersonId'].isin(multiparous['PersonId']), 'parity'] = 'Multiparous'

In [83]:
full_med['parity'] = 'Unknown'
full_med.loc[full_med['PersonId'].isin(primiparous['PersonId']), 'parity'] = 'Primiparous'
full_med.loc[full_med['PersonId'].isin(multiparous['PersonId']), 'parity'] = 'Multiparous'

In [65]:
# merge conception time w/ T2D conditions
t2d_with_conception = type2_diabet.merge(
    delivery_df_t3[['PersonId', 'estimated_conception_date']],
    on='PersonId',
    how='left'
)

# check T2D happen before preg
t2d_with_conception['pre_pregnancy_t2d'] = (
    t2d_with_conception['RecordedDateTime'] < t2d_with_conception['estimated_conception_date']
)

# checking PersonId
t2d_before_preg = (
    t2d_with_conception[t2d_with_conception['pre_pregnancy_t2d']]
    .drop_duplicates(subset='PersonId')
)

# getting ppl have T2D before preg
pre_preg_t2d_ids = t2d_before_preg['PersonId'].unique()
delivery_df_t3['t2d_before_pregnancy'] = delivery_df_t3['PersonId'].isin(pre_preg_t2d_ids)

#delivery_df_t3.t2d_before_pregnancy.value_counts()

# merge conception time w/ hypter
hyper_with_conception = hypertension.merge(
    delivery_df_t3[['PersonId', 'estimated_conception_date']],
    on='PersonId',
    how='left'
)

# check if hyper happen before preg
hyper_with_conception['pre_pregnancy_hyper'] = (
    hyper_with_conception['RecordedDateTime'] < hyper_with_conception['estimated_conception_date']
)

# find ppl diagnosie w/hyper before preg
hyper_before_preg = (
    hyper_with_conception[hyper_with_conception['pre_pregnancy_hyper']]
    .drop_duplicates(subset='PersonId')
)

# getting ids
pre_preg_hyper_ids = hyper_before_preg['PersonId'].unique()
delivery_df_t3['hyper_before_pregnancy'] = delivery_df_t3['PersonId'].isin(pre_preg_hyper_ids)

delivery_df_t3.t2d_before_pregnancy.value_counts(), delivery_df_t3.hyper_before_pregnancy.value_counts()

In [84]:
def condition_before_pregnancy(condition_df, delivery_df, condition_col_name):
    # merge with conception date
    merged = condition_df.merge(
        delivery_df[['PersonId', 'estimated_LMP']],
        on='PersonId',
        how='left'
    )

    # check if condition happened before pregnancy
    merged['pre_pregnancy'] = (
        merged['RecordedDateTime'] < merged['estimated_LMP']
    )

    # find people with condition before pregnancy
    before_preg = merged[merged['pre_pregnancy']].drop_duplicates(subset='PersonId')

    # get list of IDs
    pre_preg_ids = before_preg['PersonId'].unique()

    # add flag to delivery_df
    delivery_df[condition_col_name] = delivery_df['PersonId'].isin(pre_preg_ids)

    return delivery_df


In [100]:
delivery_df_t3 = condition_before_pregnancy(type2_diabet, delivery_df_t3, 't2d_before_pregnancy')
delivery_df_t3 = condition_before_pregnancy(hypertension, delivery_df_t3, 'hyper_before_pregnancy')

delivery_df_t3 = condition_before_pregnancy(hyperlipidemia, delivery_df_t3, 'hyperlipid')
delivery_df_t3 = condition_before_pregnancy(osa, delivery_df_t3, 'osa')
delivery_df_t3 = condition_before_pregnancy(depression, delivery_df_t3, 'depression')

delivery_df_t3.t2d_before_pregnancy.value_counts(), delivery_df_t3.hyper_before_pregnancy.value_counts()


In [85]:
full_med = condition_before_pregnancy(type2_diabet, full_med, 't2d_before_pregnancy')
full_med = condition_before_pregnancy(hypertension, full_med, 'hyper_before_pregnancy')

full_med = condition_before_pregnancy(hyperlipidemia, full_med, 'hyperlipid')
full_med = condition_before_pregnancy(osa, full_med, 'osa')
full_med = condition_before_pregnancy(depression, full_med, 'depression')

full_med.t2d_before_pregnancy.value_counts(), full_med.hyper_before_pregnancy.value_counts()


In [101]:
# have gestional diabetes but have no t2d before preg
delivery_df_t3['gest_diabetes_no_prior_t2d'] = (
    delivery_df_t3['gest_diabet'] & ~delivery_df_t3['t2d_before_pregnancy']
)

# have gestional hypertension but have no hyper before preg
delivery_df_t3['gest_hyper_no_prior_hyper'] = (
    delivery_df_t3['gest_hyper'] & ~delivery_df_t3['hyper_before_pregnancy']
)

# have preeclampsia but have no hyper before preg
delivery_df_t3['preeclampsia_no_prior_hyper'] = (
    delivery_df_t3['preeclampsia'] & ~delivery_df_t3['hyper_before_pregnancy']
)
delivery_df_t3.gest_diabetes_no_prior_t2d.value_counts(), delivery_df_t3.gest_hyper_no_prior_hyper.value_counts(), delivery_df_t3.preeclampsia_no_prior_hyper.value_counts()

In [86]:
# have gestional diabetes but have no t2d before preg
full_med['gest_diabetes_no_prior_t2d'] = (
    full_med['gest_diabet'] & ~full_med['t2d_before_pregnancy']
)

# have gestional hypertension but have no hyper before preg
full_med['gest_hyper_no_prior_hyper'] = (
    full_med['gest_hyper'] & ~full_med['hyper_before_pregnancy']
)

# have preeclampsia but have no hyper before preg
full_med['preeclampsia_no_prior_hyper'] = (
    full_med['preeclampsia'] & ~full_med['hyper_before_pregnancy']
)
full_med.gest_diabetes_no_prior_t2d.value_counts(), full_med.gest_hyper_no_prior_hyper.value_counts(), full_med.preeclampsia_no_prior_hyper.value_counts()

In [102]:
delivery_df_t3.preeclampsia_no_prior_hyper.value_counts()

In [ ]:
med_delivery_df_t3 = delivery_df_t3[delivery_df_t3.source_type == 'med']
print(len(med_delivery_df_t3))
#med_delivery_df_t3.head()
print(med_delivery_df_t3.preeclampsia.value_counts(), med_delivery_df_t3.csection.value_counts(),
med_delivery_df_t3.large_gest_age.value_counts(),med_delivery_df_t3.small_gest_age.value_counts(),med_delivery_df_t3.intra_grow_restrict.value_counts(),
med_delivery_df_t3.gest_diabetes_no_prior_t2d.value_counts(), med_delivery_df_t3.gest_hyper_no_prior_hyper.value_counts())

In [103]:
delivery_df_t3 = delivery_df_t3[
    ~delivery_df_t3["PersonId"].isin(stillbirth["PersonId"])
]
delivery_df_t3["PersonId"].nunique()

In [87]:
full_med = full_med[
    ~full_med["PersonId"].isin(stillbirth["PersonId"])
]
full_med["PersonId"].nunique()

In [104]:
print(delivery_df_t3.PersonId.nunique())

### Getting Person Info, DOB, age, income, race

In [88]:
df = ps.sql("SELECT * FROM Person").to_pandas()
person = decode_concepts(df)
#person.head()
df = ps.sql("SELECT * FROM PersonRace").to_pandas()
race = decode_concepts(df)
#race.head()
len(person.Id.unique()),len(person.Id)

In [106]:
person = person.rename(columns={'Id': 'PersonId'})
delivery_df_t3 = delivery_df_t3.merge(person[['PersonId','BirthDateTime','Ethnicity','Gender']], on='PersonId', how='left')
delivery_df_t3 = delivery_df_t3.merge(race[['PersonId','Race']], on='PersonId', how='left')
#delivery_df_t3.head()

In [89]:
person = person.rename(columns={'Id': 'PersonId'})
full_med = full_med.merge(person[['PersonId','BirthDateTime','Ethnicity','Gender']], on='PersonId', how='left')
full_med = full_med.merge(race[['PersonId','Race']], on='PersonId', how='left')
#delivery_df_t3.head()

In [107]:
# age calucation 
def calculate_age(event_date, dob):
    return (event_date - dob).days / 365.25

# change to datetime.date
delivery_df_t3['delivery_date'] = pd.to_datetime(delivery_df_t3['delivery_date']).dt.date
delivery_df_t3['BirthDateTime'] = pd.to_datetime(delivery_df_t3['BirthDateTime']).dt.date
delivery_df_t3['event_date'] = pd.to_datetime(delivery_df_t3['event_date']).dt.date

# assume merged have DOB、surgery_date、med_date、RecordedDateTime
delivery_df_t3['age_at_delivery'] = delivery_df_t3.apply(lambda row: calculate_age(row['delivery_date'], row['BirthDateTime']), axis=1)
#first take medication and surgery
delivery_df_t3['age_at_event'] = delivery_df_t3.apply(lambda row: calculate_age(row['event_date'], row['BirthDateTime']), axis=1)
#delivery_df_t3.head()

In [90]:
# age calucation 
def calculate_age(event_date, dob):
    return (event_date - dob).days / 365.25

# change to datetime.date
full_med['delivery_date'] = pd.to_datetime(full_med['delivery_date']).dt.date
full_med['BirthDateTime'] = pd.to_datetime(full_med['BirthDateTime']).dt.date
full_med['event_date'] = pd.to_datetime(full_med['event_date']).dt.date

# assume merged have DOB、surgery_date、med_date、RecordedDateTime
full_med['age_at_delivery'] = full_med.apply(lambda row: calculate_age(row['delivery_date'], row['BirthDateTime']), axis=1)
#first take medication and surgery
full_med['age_at_event'] = full_med.apply(lambda row: calculate_age(row['event_date'], row['BirthDateTime']), axis=1)
#delivery_df_t3.head()

In [ ]:
delivery_df_t3.age_at_delivery.median(), delivery_df_t3.age_at_event.median(),
delivery_df_t3[delivery_df_t3.source_type == 'med'].age_at_delivery.median(), delivery_df_t3[delivery_df_t3.source_type == 'med'].age_at_event.median()
delivery_df_t3[delivery_df_t3.source_type == 'surgery'].age_at_delivery.median(), delivery_df_t3[delivery_df_t3.source_type == 'surgery'].age_at_event.median()

In [108]:
delivery_df_t3.Race.value_counts()

In [109]:
delivery_df_t3.Ethnicity.value_counts()

In [91]:
def combine_race_ethnicity(row):
    race = str(row['Race']).strip().lower()
    ethnicity = str(row['Ethnicity']).strip().lower()

    if pd.isna(race) or pd.isna(ethnicity):
        return 'Unknown'

    if ethnicity == 'hispanic or latino':
        return 'Hispanic'

    elif ethnicity == 'not hispanic or latino':
        if race == 'white':
            return 'Non-Hispanic White'
        elif race == 'black or african american':
            return 'Non-Hispanic Black'
        elif race in ['asian', 'american indian or alaska native',
                      'native hawaiian or other pacific islander', 'other race']:
            return 'Other'
        else:
            return 'Unknown'

    else:
        return 'Unknown'


In [111]:
delivery_df_t3['race_ethnicity'] = delivery_df_t3.apply(combine_race_ethnicity, axis=1)

In [92]:
full_med['race_ethnicity'] = full_med.apply(combine_race_ethnicity, axis=1)

In [112]:
delivery_df_t3.race_ethnicity.value_counts()

In [113]:
delivery_df_t3.head()

##### adding income

In [93]:
df = ps.sql("SELECT * FROM SocialDeterminantsOfHealth").to_pandas()
soh = decode_concepts(df)
soh.head()

In [94]:
dd = snapshot.get_data_dictionary()
dd.loc[dd['table'] == 'SocialDeterminantsOfHealth']

sql = """
SELECT 
* 
FROM SocialDeterminantsOfHealth
"""
SDOH = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_sdoh')

sql = """
SELECT 
* 
FROM Concept
"""

concept = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_concept')


# concept.head()

sql = """
SELECT *
FROM tbl_concept
WHERE ConceptClass = 'SDOHAttribute';
"""

SDOH_concepts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_concepts')

# SDOH_concepts

sql = """
SELECT ConceptId, ConceptName
FROM tbl_concept;
"""

concept_sel = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_concept_sel')

# concept_sel.head()

# Spark SQL option

sql = """
SELECT
    agg.AttributeConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         AttributeConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         AttributeConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.AttributeConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

attribute_counts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_attribute_counts')

attribute_counts_sorted = attribute_counts\
 .sort_values(by='count', ascending=False)

# display(attribute_counts_sorted)

In [95]:

sql = """
SELECT
    agg.NormalizedValueConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         NormalizedValueConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         NormalizedValueConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.NormalizedValueConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

normalized_value_counts = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_anormalized_value_counts')

normalized_value_counts_sorted = normalized_value_counts.\
   sort_values(by='count', ascending=False)

normalized_value_counts_sorted.head(10)

In [96]:
# Pandas on Spark option

# source_concept_id = SDOH.groupby('SourceConceptId').size().reset_index().copy()
# source_concept_id = source_concept_id.merge(concept, how = 'left', 
#                                                   left_on = 'SourceConceptId',
#                                                   right_on = 'ConceptId').sort_values(0, ascending=False)
                                                  

# source_concept_id.head(10)

#############################################################
#############################################################

# Spark SQL option

sql = """
SELECT
    agg.SourceConceptId,
    agg.count,
    c.*
FROM
    (SELECT
         SourceConceptId,
         COUNT(*) as count
     FROM
         tbl_sdoh
     GROUP BY
         SourceConceptId) agg
LEFT JOIN
    tbl_concept_sel c
ON
    agg.SourceConceptId = c.ConceptId
ORDER BY
    agg.count DESC
"""

source_concept_counts = snapshot.load_sql_table(sql, cache = True, 
   view_name = 'tbl_source_concept_counts')

source_concept_counts_sorted = source_concept_counts\
   .sort_values(by='count', ascending=False)

source_concept_counts_sorted.head()

In [97]:
sql = """
SELECT 
    PersonId, 
    EffectiveStartDateTime, 
    AttributeConceptId, 
    NormalizedValueNumeric, 
    NormalizedValueConceptId
FROM 
    tbl_sdoh;
"""

SDOH_2 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_2')
   
# SDOH_2.head(3)

sql = """
SELECT
    tbl_sdoh_2.PersonId,
    tbl_sdoh_2.EffectiveStartDateTime,
    tbl_sdoh_2.NormalizedValueNumeric,
    tbl_sdoh_2.NormalizedValueConceptId,
    tbl_concept_sel.ConceptId,
    tbl_concept_sel.ConceptName AS Attribute
FROM
    tbl_sdoh_2
LEFT JOIN tbl_concept_sel ON
    tbl_sdoh_2.AttributeConceptId = tbl_concept_sel.ConceptId;
"""

SDOH_3 = snapshot.load_sql_table(sql, cache = True, view_name = 'tbl_sdoh_3')
# SDOH_3.head(3)

sql = """
SELECT
    s.PersonId,
    s.EffectiveStartDateTime,
    s.Attribute,
    c.ConceptName AS Attribute_Value_Categorical,
    s.NormalizedValueNumeric AS Attribute_Value_Numeric
FROM
    tbl_sdoh_3 s
LEFT JOIN tbl_concept_sel c ON
    s.NormalizedValueConceptId = c.ConceptId;

"""

SDOH_4 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_4')
   
#SDOH_4.head(3)

sql = """
WITH SortedData AS (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY PersonId, 
           Attribute ORDER BY EffectiveStartDateTime DESC, Attribute) 
             AS rn
    FROM tbl_sdoh_4
)
SELECT *
FROM SortedData
WHERE rn = 1
ORDER BY PersonId, EffectiveStartDateTime, Attribute;
"""

SDOH_5 = snapshot.load_sql_table(sql, 
   cache = True, view_name = 'tbl_sdoh_5')
   
SDOH_5.head(10)

In [98]:
SDOH_5_p = SDOH_5.to_pandas()

SDOH_wide = SDOH_5_p.pivot(index='PersonId', columns='Attribute', 
   values=['EffectiveStartDateTime', 'Attribute_Value_Categorical', 
   'Attribute_Value_Numeric']).reset_index()


SDOH_wide.columns = SDOH_wide.columns.map(lambda index: f'{index[0]}_{index[1]}')
SDOH_wide.rename({'PersonId_': 'PersonId'}, axis=1, inplace=True)

#SDOH_wide.head(10)
person_p = person.copy()

person_SDOH = person_p.merge(SDOH_wide, 
    how = 'left', 
    left_on = 'PersonId', 
    right_on = 'PersonId')

# person_SDOH.head()

In [ ]:
person_SDOH.Attribute_Value_Categorical_HouseholdAnnualIncomeRange.value_counts()

In [121]:
delivery_df_t3 = delivery_df_t3.merge(person_SDOH[['PersonId','Attribute_Value_Categorical_EstimatedAnnualIncome']], on='PersonId', how='left')
delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].fillna('Unknown')

In [99]:
full_med = full_med.merge(person_SDOH[['PersonId','Attribute_Value_Categorical_EstimatedAnnualIncome']], on='PersonId', how='left')
full_med['Attribute_Value_Categorical_EstimatedAnnualIncome'] = full_med['Attribute_Value_Categorical_EstimatedAnnualIncome'].fillna('Unknown')

In [100]:
## changing income interval
def reclassify_income(bracket):
    try:
        low = int(bracket.split('-')[0].replace(',', '').strip())
        if low <= 50000:
            return '≤50000'
        elif low <= 80000:
            return '50001-80000'
        else:
            return '>80000'
    except:
        return 'Unknown'

# delivery_df_t3['Income'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].apply(reclassify_income)
# # delivery_df_t3['pretreatment_weight'] = delivery_df_t3['pretreatment_weight'].fillna('Unknown')
# delivery_df_t3.head()

full_med['Income'] = full_med['Attribute_Value_Categorical_EstimatedAnnualIncome'].apply(reclassify_income)


In [104]:
# we decided to use the Household Income Range
# delivery_df_t3 = delivery_df_t3.merge(person_SDOH[['PersonId','Attribute_Value_Categorical_EstimatedAnnualIncome']], on='PersonId', how='left')
# delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].fillna('Unknown')
# delivery_df_t3['EstimatedIncome'] = delivery_df_t3['Attribute_Value_Categorical_EstimatedAnnualIncome'].apply(reclassify_income)

In [124]:
delivery_df_t3['weight_loss'] = (
    delivery_df_t3['pretreatment_weight'] - delivery_df_t3['prepreg_weight'])

### Sub-Analysis Start Medication within 1 year of preg

In [125]:
delivery_df_t3['Race'] = delivery_df_t3['Race'].fillna('Unknown')
med_delivery_df_t3 = delivery_df_t3[delivery_df_t3.source_type == 'med']
print(len(med_delivery_df_t3))
med_delivery_df_t3.head()

In [127]:
def check_event_in_past_year(row, df):
    start = row['estimated_LMP'] - pd.DateOffset(years=1)
    end = row['']
    person_events = df[df['PersonId'] == row['PersonId']]['event_date']
    return ((person_events >= start) & (person_events < end)).any()

med_delivery_df_t3['event_in_past_year'] = med_delivery_df_t3.apply(lambda row: check_event_in_past_year(row, med_delivery_df_t3), axis=1)

In [128]:
med_delivery_df_t3.head()

In [129]:
med_delivery_df_t3.event_in_past_year.value_counts()

### Creating Table 1

In [130]:
from statsmodels.formula.api import ols
! pip install tableone
from tableone import TableOne
import statsmodels.api as sm

In [138]:
delivery_df_t3 = delivery_df_t3.merge(pretreat_df, on='PersonId', how='left')

In [139]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/test_t3.csv"
delivery_df_t3.to_csv(file_to_write, index = False)

In [52]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/weights_test.csv"
all_weights_df_pd.to_csv(file_to_write, index = False)

In [101]:
file_to_write = output_path_local + "/full_med_wo_weight.csv"
full_med.to_csv(file_to_write, index = False)

In [140]:
table1 = delivery_df_t3.copy()
table1.columns

In [141]:
# readin table t3
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/test_t3.csv"
table1 = pd.read_csv(file_to_read)
table1.head()

##### Table 1 with Med/Surgery Group Separate - P-value

In [142]:
t2 = table1[['PersonId', 'source_type', 'gestational_week',
       'gestational_age_days_at_delivery', 'preterm', 'has_prepreg_weight',
       'prepreg_weight', 'has_predelivery_weight', 'predelivery_weight',
       'pretreatment_weight', 'months_between_event_and_conception',
       'has_both_weights', 'gestation_weight', 'PrePregnancyBMI',
       'nearDeliveryBMI', 'gest_diabet', 'gest_hyper', 'preeclampsia',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'obstetric_care', 'prior_Csection', 'Prior_Preterm_Birth',
       'delivery_type', 'infant_gest_age_class', 'parity',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'hyperlipid', 'osa',
       'depression', 'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper',
       'preeclampsia_no_prior_hyper', 'Ethnicity', 'Gender',
       'Race', 'age_at_delivery', 'age_at_event', 'race_ethnicity',
       'Attribute_Value_Categorical_EstimatedAnnualIncome', 'Income', 'preTreatmentBMI',
       'weight_loss']]

columns_df = ['preterm',
       'prepreg_weight', 'predelivery_weight', 'pretreatment_weight',
       'months_between_event_and_conception', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'preeclampsia',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict', 'obstetric_care', 'prior_Csection', 'Prior_Preterm_Birth',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity', 'preTreatmentBMI',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity', 'age_at_delivery',
       'age_at_event',
       'Income']

groupby = 'source_type'

cate_df = ['preterm', 'preeclampsia',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity', 'obstetric_care', 'prior_Csection', 'Prior_Preterm_Birth',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper', 'race_ethnicity',
       'Income']


table1_p = TableOne(t2, columns=columns_df, categorical=cate_df, groupby = groupby, pval=True)
#table1_p

In [ ]:
file_to_write = output_path_local + "/results/table1pvalue.html"
table1_p.to_html(file_to_write, index = True)

In [143]:
table1_p

#### NO NEED TO RUN THIS FOLLOWING CHUNK, I was just trying to see the p-value difference but it is not filtering correctly

In [ ]:
table1_t2d = table1.copy()
table1_t2d.loc[(table1_t2d['t2d_before_pregnancy'] == False)&(table1_t2d['source_type'] == 'med'), 'source_type'] = 'med_wo_t2d'
t3 = table1_t2d[['PersonId', 'source_type', 'is_preterm',
       'prepreg_weight', 'predelivery_weight', 'pretreatment_weight', 'preTreatmentBMI',
       'months_between_event_and_conception', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'preeclampsia',
       'csection', 'excessive_fetal_weight','intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity', 
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity', 'age_at_delivery',
       'age_at_event',
       'Income']]
columns_df = ['is_preterm',
       'prepreg_weight', 'predelivery_weight', 'pretreatment_weight',
       'months_between_event_and_conception', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'preeclampsia', 'preTreatmentBMI',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity', 'age_at_delivery',
       'age_at_event',
       'Income']

groupby = 'source_type'

cate_df = ['is_preterm', 'preeclampsia',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper', 'race_ethnicity',
       'Income']


table1_p3 = TableOne(t3, columns=columns_df, categorical=cate_df, groupby = groupby, pval=True)
#table1_p

In [ ]:
table1_p3

#### Medication group table 1

In [ ]:
table1 = med_delivery_df_t3.copy()
table1 = med_delivery_df_t3[med_delivery_df_t3.t2d_before_pregnancy == False].copy()
t1 = table1[['PersonId', 'is_preterm', 'event_in_past_year', 
       'prepreg_weight', 'predelivery_weight', 'pretreatment_weight',
       'months_between_event_and_conception', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'preeclampsia',
       'csection', 'excessive_fetal_weight','intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity', 
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity', 'age_at_delivery',
       'age_at_event',
       'Income']]
columns_df = ['is_preterm', 'event_in_past_year',
       'prepreg_weight', 'predelivery_weight', 'pretreatment_weight',
       'months_between_event_and_conception', 'weight_loss',
       'gestation_weight', 'PrePregnancyBMI', 'nearDeliveryBMI', 'preeclampsia',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity', 'age_at_delivery',
       'age_at_event',
       'Income']

cate_df = ['is_preterm', 'preeclampsia', 'event_in_past_year',
       'csection', 'excessive_fetal_weight', 'intra_grow_restrict',
       'hyperlipid', 'osa', 'depression', 'delivery_type', 'infant_gest_age_class',
       't2d_before_pregnancy', 'hyper_before_pregnancy', 'parity',
       'gest_diabetes_no_prior_t2d', 'gest_hyper_no_prior_hyper', 'preeclampsia_no_prior_hyper',
       'race_ethnicity',
       'Income']

# groupby = 'event_in_past_year'

table1 = TableOne(t1, columns=columns_df, categorical=cate_df,  pval=False) #groupby = groupby, pval=True)
#table1

In [ ]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/results/medtable1_not2d.html"
table1.to_html(file_to_write, index = True)
table1

In [14]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/medication_full.csv"
medication_full.to_csv(file_to_write, index = False)

### Medication Discontinuous - Don't Run see Notebook ''drug_use_metrics_heilbrunn_analysis''

In [ ]:
medication_df = medication_full.copy()
medication_full.Code.value_counts()

In [ ]:
medication_df['med_date'] = pd.to_datetime(medication_df['med_date'])
medication_df = medication_df.sort_values(['PersonId', 'med_date'])
medication_df = medication_df.groupby(['PersonId', 'med_date', 'Code'], as_index=False).agg({'DaysSupply': 'sum'})
medication_df['EndDate'] = medication_df['med_date'] + pd.to_timedelta(medication_df['DaysSupply'], unit='d')

In [ ]:
medication_df['NextStart'] = medication_df.groupby('PersonId')['med_date'].shift(-1)
medication_df['GapDays'] = (medication_df['NextStart'] - medication_df['EndDate']).dt.days

person_ranges = medication_df.groupby('PersonId').agg(
    IndexDate=('med_date', 'min'),
    LastEndDate=('EndDate', 'max')
).reset_index()
person_ranges['TotalTime'] = (person_ranges['LastEndDate'] - person_ranges['IndexDate']).dt.days
person_ranges.head()

In [ ]:
long_gaps = medication_df[medication_df['GapDays'] >= 60].copy()

gap_summary = long_gaps.groupby('PersonId').agg(
    GapTime=('GapDays', 'sum'),
    Discontinued_60=('GapDays', lambda x: len(x) > 0)
).reset_index()

feature_table = person_ranges.merge(gap_summary, on='PersonId', how='left')
feature_table['GapTime'] = feature_table['GapTime'].fillna(0)
feature_table['Discontinued_60'] = feature_table['Discontinued_60'].fillna(False)
feature_table['NetMedicationTime'] = feature_table['TotalTime'] - feature_table['GapTime']

In [ ]:
feature_table['TotalTime_months'] = (feature_table['TotalTime'] / 30).round(2)
feature_table['GapTime_months'] = (feature_table['GapTime'] / 30).round(2)
feature_table['NetMedicationTime_months'] = (feature_table['NetMedicationTime'] / 30).round(2)


In [ ]:
feature_table.columns

In [ ]:
feature_table.head()

In [ ]:
# feature_table['match_delivery'] = medication_full['PersonId'].isin(feature_table['PersonId'])
# feature_table.head()
feature_table = feature_table[feature_table['PersonId'].isin(delivery_df_t3['PersonId'])]
table1 = feature_table.copy()
t1 = feature_table.copy()
columns_df = ['Discontinued_60', 'TotalTime_months',
       'GapTime', 'NetMedicationTime_months']

cate_df = ['Discontinued_60']

table1 = TableOne(t1, columns=columns_df, categorical=cate_df, pval=False)
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/medtable1.html"
table1.to_html(file_to_write, index = True)
table1

In [ ]:
feature_table.match_delivery.value_counts()

In [ ]:
medication_full[medication_full.PersonId == '03a45ffa-c6a4-f663-0032-27c3eb13e04b']